# Chapter 4: Categorical Encoding

> **Data** hiking.csv  (exercises use clinic_referrals.csv) · **Time** 45 min · **Needs first** C1, C3

## What you will be able to do

1. Tell an ordinal column from a nominal one and encode each correctly
2. Apply LabelEncoder, OneHotEncoder, OrdinalEncoder and pd.get_dummies
3. State what one-hot encoding costs on a high-cardinality column
4. Apply frequency and target encoding, and name the leakage risk in target encoding
5. Handle a category present in test data but absent from training

## The one idea to hold on to

Encoding a category as a number tells the model those numbers can be compared. If they cannot, you have added a false fact rather than a feature.

Machine learning algorithms cannot understand text categories directly - they require numerical input. **Categorical Encoding** is the process of converting categorical (text) variables into numerical format.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

# Set display options
pd.set_option('display.max_columns', None)
%matplotlib inline

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Understanding Categorical Variables

Categorical variables fall into two main types:

| Type | Definition | Examples | Encoding Methods |
|------|------------|----------|------------------|
| **Nominal** | No inherent order | Colors, Countries, Names | One-Hot, Frequency, Target |
| **Ordinal** | Has natural order | Low/Medium/High, Grades | Ordinal, Label |

### Why Does This Matter?

Using the wrong encoding can mislead your model:
- If you use Label Encoding (1, 2, 3) for nominal data like colors, the model might think Red=1 is "less than" Blue=3
- This creates false mathematical relationships that don't exist in reality

In [2]:
# Load the hiking dataset
df = pd.read_csv('../../00-data/hiking.csv')

# Fill missing values for demonstration
df['Difficulty'] = df['Difficulty'].fillna('Unknown')

print("Dataset Shape:", df.shape)
print("\nCategorical Columns:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col}: {df[col].nunique()} unique values")

Dataset Shape: (33, 11)

Categorical Columns:
  Prop_ID: 19 unique values
  Name: 33 unique values
  Location: 30 unique values
  Park_Name: 19 unique values
  Length: 19 unique values
  Difficulty: 7 unique values
  Other_Details: 27 unique values
  Accessible: 2 unique values
  Limited_Access: 2 unique values


---

## 2. Binary/Label Encoding

**Binary Encoding** converts Yes/No or True/False columns to 1/0.

**Label Encoding** assigns a unique integer to each category (0, 1, 2, 3...).

**When to Use:**
- Binary features (only 2 values)
- Ordinal features where order matters

**When NOT to Use:**
- Nominal features with no order (use One-Hot instead)

![Label Encoding](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmiro.medium.com%2Fv2%2Fresize%3Afit%3A1358%2F1*fKzvuNxDRQKUT6D7BG9FBQ.png)

In [3]:
# Check binary columns
print("Accessible column values:", df['Accessible'].unique())
print("Limited_Access column values:", df['Limited_Access'].unique())

Accessible column values: ['Y' 'N']
Limited_Access column values: ['N' 'Y']


In [4]:
# Method 1: Manual Mapping (preferred for binary)
df['Accessible_Enc'] = df['Accessible'].map({'Y': 1, 'N': 0})
df['Limited_Access_Enc'] = df['Limited_Access'].map({'Y': 1, 'N': 0})

print("Binary Encoding Results:")
df[['Accessible', 'Accessible_Enc', 'Limited_Access', 'Limited_Access_Enc']].head()

Binary Encoding Results:


,Accessible,Accessible_Enc,Limited_Access,Limited_Access_Enc
0,Y,1,N,0
1,N,0,N,0
2,N,0,N,0
3,N,0,N,0
4,N,0,N,0


In [5]:
# Method 2: Using sklearn LabelEncoder
le = LabelEncoder()

# Fit and transform
df['Accessible_LE'] = le.fit_transform(df['Accessible'])

# View the mapping
print("LabelEncoder Mapping:")
for i, label in enumerate(le.classes_):
    print(f"  '{label}' -> {i}")

LabelEncoder Mapping:
  'N' -> 0
  'Y' -> 1


---

## 3. One-Hot Encoding

One-Hot Encoding creates a **new binary column for each category**. Each row has exactly one '1' and the rest are '0'.

**When to Use:**
- Nominal features (no inherent order)
- Low to medium cardinality (< 15-20 categories)
- Linear models, neural networks

**When NOT to Use:**
- High cardinality features (creates too many columns)
- Tree-based models (can handle label encoding)

![One-Hot Encoding](https://miro.medium.com/max/1400/0*T5jaa2othYfXZX9W.)

In [6]:
# Check Park_Name - this is a nominal categorical variable
print(f"Number of unique parks: {df['Park_Name'].nunique()}")
print("\nPark name counts:")
print(df['Park_Name'].value_counts())

Number of unique parks: 19

Park name counts:
Park_Name
La Tourette Parks & Golf Course     5
Van Cortlandt Park                  5
Prospect Park                       4
Forest Park                         3
High Rock Park                      2
Marine Park                         1
Wolfes Pond Park                    1
Bronx Park                          1
Inwood Hill Park                    1
Pelham Bay Park                     1
Clove Lakes Park                    1
Long Pond Park                      1
Deere Park                          1
Arden Woods                         1
Willowbrook Park                    1
William T. Davis Wildlife Refuge    1
Cunningham Park                     1
Alley Pond Park                     1
Conference House Park               1
Name: count, dtype: int64


In [7]:
# Method 1: Using pandas get_dummies (easiest)
park_dummies = pd.get_dummies(df['Park_Name'], prefix='Park')

print(f"Original columns: 1 (Park_Name)")
print(f"New columns after One-Hot: {park_dummies.shape[1]}")
print("\nSample of encoded data:")
park_dummies.head(3)

Original columns: 1 (Park_Name)
New columns after One-Hot: 19

Sample of encoded data:


,Park_Alley Pond Park,Park_Arden Woods,Park_Bronx Park,Park_Clove Lakes Park,Park_Conference House Park,Park_Cunningham Park,Park_Deere Park,Park_Forest Park,Park_High Rock Park,Park_Inwood Hill Park,Park_La Tourette Parks & Golf Course,Park_Long Pond Park,Park_Marine Park,Park_Pelham Bay Park,Park_Prospect Park,Park_Van Cortlandt Park,Park_William T. Davis Wildlife Refuge,Park_Willowbrook Park,Park_Wolfes Pond Park
0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False


In [8]:
# Combine with original dataframe
df_encoded = pd.concat([df, park_dummies], axis=1)
print(f"Original shape: {df.shape}")
print(f"After One-Hot: {df_encoded.shape}")

Original shape: (33, 14)
After One-Hot: (33, 33)


### Handling the Dummy Variable Trap

When using One-Hot Encoding with linear models, you should drop one category to avoid **perfect multicollinearity** (the dummy variable trap).

In [9]:
# Drop first category to avoid multicollinearity
park_dummies_dropped = pd.get_dummies(df['Park_Name'], prefix='Park', drop_first=True)

print(f"With drop_first=True: {park_dummies_dropped.shape[1]} columns")
print(f"Without drop_first:   {park_dummies.shape[1]} columns")

With drop_first=True: 18 columns
Without drop_first:   19 columns


---

## 4. Ordinal Encoding

Ordinal Encoding assigns integers based on a **meaningful order** that you define.

**When to Use:**
- Features with natural ordering (Low < Medium < High)
- Survey responses (Disagree < Neutral < Agree)
- Education levels, size categories

**Key Difference from Label Encoding:**
- Label Encoding: arbitrary order (alphabetical)
- Ordinal Encoding: YOU define the meaningful order

![Ordinal Encoding](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F4752d819-8d28-4f62-a2f8-ab07af9ed48e_1431x702.jpeg)

In [10]:
# Check Difficulty column - this has natural ordering
print("Difficulty values:")
print(df['Difficulty'].value_counts())

Difficulty values:
Difficulty
Easy                  12
Unknown                8
Easy/Moderate          6
Moderate               3
Moderate/Difficult     2
Various                1
Easy                   1
Name: count, dtype: int64


In [11]:
# Method 1: Manual Mapping (explicit and clear)
difficulty_order = {
    'Unknown': 0,
    'Easy': 1,
    'Easy ': 1,  # Handle whitespace variant
    'Easy/Moderate': 2,
    'Moderate': 3,
    'Moderate/Difficult': 4,
    'Various': 2  # Treating as medium difficulty
}

df['Difficulty_Ordinal'] = df['Difficulty'].map(difficulty_order)

print("Ordinal Encoding Results:")
df[['Difficulty', 'Difficulty_Ordinal']].drop_duplicates().sort_values('Difficulty_Ordinal')

Ordinal Encoding Results:


,Difficulty,Difficulty_Ordinal
0,Unknown,0
1,Easy,1
24,Easy,1
5,Various,2
10,Easy/Moderate,2
17,Moderate,3
14,Moderate/Difficult,4


In [12]:
# Method 2: Using sklearn OrdinalEncoder
# Define the order explicitly
categories = [['Unknown', 'Easy', 'Easy ', 'Easy/Moderate', 'Various', 'Moderate', 'Moderate/Difficult']]

oe = OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1)

# Fit and transform
df['Difficulty_OE'] = oe.fit_transform(df[['Difficulty']])

print("OrdinalEncoder Results:")
df[['Difficulty', 'Difficulty_OE']].drop_duplicates()

OrdinalEncoder Results:


,Difficulty,Difficulty_OE
0,Unknown,0.0
1,Easy,1.0
5,Various,4.0
10,Easy/Moderate,3.0
14,Moderate/Difficult,6.0
17,Moderate,5.0
24,Easy,2.0


---

## 5. Frequency Encoding

Frequency Encoding replaces each category with **how often it appears** in the dataset (count or proportion).

**When to Use:**
- High cardinality features (many unique values)
- When category frequency might be predictive
- To avoid creating too many columns with One-Hot

**When NOT to Use:**
- When multiple categories have the same frequency (loses distinction)
- When frequency is not meaningful for prediction

![Frequency Encoding](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fblogger.googleusercontent.com%2Fimg%2Fb%2FR29vZ2xl%2FAVvXsEjMWhxUnQACFl_XMleDp2C0C6OzIS3N-zM58XnaIaTvbaCpZaBLHyVY_3WCr7h1DNEndg9V75cYVRxqFn06tajK_DcuuN4NAY2277Zfv5DM4VUxwdLmfUFPITaDdNgAviPhfJ8nLS7Pv0bZTmYji7u1CC2c6Y0E1f7Gy7m0rnQShdJxUHe4KFG6Cn3CUQ%2Fs1222%2Ffrequency-encoding.jpg)

In [13]:
# Calculate frequency (count) for each park
freq_map = df['Park_Name'].value_counts().to_dict()

# Apply frequency encoding
df['Park_Freq'] = df['Park_Name'].map(freq_map)

print("Frequency Encoding:")
df[['Park_Name', 'Park_Freq']].drop_duplicates().sort_values('Park_Freq', ascending=False).head(10)

Frequency Encoding:


,Park_Name,Park_Freq
10,La Tourette Parks & Golf Course,5
20,Van Cortlandt Park,5
1,Prospect Park,4
6,Forest Park,3
17,High Rock Park,2
0,Marine Park,1
26,Wolfes Pond Park,1
31,Bronx Park,1
30,Inwood Hill Park,1
29,Pelham Bay Park,1


In [14]:
# Alternatively, use proportion instead of count
prop_map = (df['Park_Name'].value_counts() / len(df)).to_dict()
df['Park_Proportion'] = df['Park_Name'].map(prop_map)

print("Proportion Encoding (sums to 1):")
print(f"Sum of proportions: {df['Park_Proportion'].sum():.4f}")
df[['Park_Name', 'Park_Proportion']].drop_duplicates().head()

Proportion Encoding (sums to 1):
Sum of proportions: 2.8182


,Park_Name,Park_Proportion
0,Marine Park,0.030303
1,Prospect Park,0.121212
5,Alley Pond Park,0.030303
6,Forest Park,0.090909
9,Cunningham Park,0.030303


---

## 6. Target/Mean Encoding (Advanced)

Target Encoding replaces each category with the **mean of the target variable** for that category.

**When to Use:**
- High cardinality features
- Strong relationship between category and target

**CAUTION - Data Leakage Risk:**
- Target encoding can cause **data leakage** if not done carefully
- Always fit on training data only, then transform test data
- Consider using smoothing/regularization to handle rare categories

In [15]:
# Create a synthetic target for demonstration
np.random.seed(42)
df['Trail_Rating'] = np.random.uniform(3, 5, len(df))  # Random rating 3-5

# Target encoding: mean rating per park
target_means = df.groupby('Park_Name')['Trail_Rating'].mean().to_dict()
df['Park_Target_Enc'] = df['Park_Name'].map(target_means)

print("Target Encoding (Mean Rating per Park):")
df[['Park_Name', 'Trail_Rating', 'Park_Target_Enc']].head(10)

Target Encoding (Mean Rating per Park):


,Park_Name,Trail_Rating,Park_Target_Enc
0,Marine Park,3.749080,3.749080
1,Prospect Park,4.901429,4.218693
2,Prospect Park,4.463988,4.218693
3,Prospect Park,4.197317,4.218693
4,Prospect Park,3.312037,4.218693
5,Alley Pond Park,3.311989,3.311989
6,Forest Park,3.116167,4.016917
7,Forest Park,4.732352,4.016917
8,Forest Park,4.202230,4.016917
9,Cunningham Park,4.416145,4.416145


### Preventing Data Leakage with Target Encoding

```python
# CORRECT way - fit on train, transform on test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Calculate means from TRAINING data only
target_means = X_train.groupby('Category')['Target'].mean().to_dict()

# Apply to both train and test
X_train['Cat_Encoded'] = X_train['Category'].map(target_means)
X_test['Cat_Encoded'] = X_test['Category'].map(target_means)  # Uses train means!

# Handle unseen categories in test
global_mean = y_train.mean()
X_test['Cat_Encoded'] = X_test['Cat_Encoded'].fillna(global_mean)
```

---

## 7. Choosing the Right Encoding Method

### Decision Guide

```
                        Is the feature ORDINAL (has order)?
                              /                  \
                            Yes                  No (Nominal)
                             |                    |
                      Ordinal Encoding      How many categories?
                                              /           \
                                         < 15           > 15
                                           |             |
                                    One-Hot Encoding   Frequency or
                                                       Target Encoding
```

### Summary Table

| Encoding Method | When to Use | Pros | Cons |
|-----------------|-------------|------|------|
| **Label/Binary** | Binary features, ordinal with 2 values | Simple, no dimension increase | Implies false order for nominal |
| **One-Hot** | Nominal, low cardinality | No false ordering | Increases dimensions |
| **Ordinal** | Features with meaningful order | Preserves order, no dimension increase | Requires domain knowledge |
| **Frequency** | High cardinality | Single column | Same frequency = same encoding |
| **Target** | High cardinality, supervised | Captures target relationship | Data leakage risk |

In [16]:
# Summary of all encodings we created
encoding_columns = ['Accessible', 'Accessible_Enc', 'Difficulty', 'Difficulty_Ordinal', 
                    'Park_Name', 'Park_Freq']

print("Summary of Encoded Features:")
df[encoding_columns].head(10)

Summary of Encoded Features:


,Accessible,Accessible_Enc,Difficulty,Difficulty_Ordinal,Park_Name,Park_Freq
0,Y,1,Unknown,0,Marine Park,1
1,N,0,Easy,1,Prospect Park,4
2,N,0,Easy,1,Prospect Park,4
3,N,0,Easy,1,Prospect Park,4
4,N,0,Easy,1,Prospect Park,4
5,N,0,Various,2,Alley Pond Park,1
6,N,0,Unknown,0,Forest Park,3
7,N,0,Unknown,0,Forest Park,3
8,N,0,Unknown,0,Forest Park,3
9,N,0,Unknown,0,Cunningham Park,1


---

## 8. Handling High Cardinality

When a feature has many unique values (high cardinality), consider these strategies:

| Strategy | Description |
|----------|-------------|
| **Group Rare Categories** | Combine categories with few samples into "Other" |
| **Frequency Encoding** | Replace with count/proportion |
| **Target Encoding** | Replace with target mean (with caution) |
| **Feature Hashing** | Hash categories to fixed number of bins |

In [17]:
# Example: Group rare categories into 'Other'
# Keep only parks with more than 1 trail, group rest as 'Other'
park_counts = df['Park_Name'].value_counts()
rare_parks = park_counts[park_counts <= 1].index

df['Park_Grouped'] = df['Park_Name'].apply(lambda x: 'Other' if x in rare_parks else x)

print(f"Original unique parks: {df['Park_Name'].nunique()}")
print(f"After grouping rare: {df['Park_Grouped'].nunique()}")
print("\nGrouped value counts:")
print(df['Park_Grouped'].value_counts())

Original unique parks: 19
After grouping rare: 6

Grouped value counts:
Park_Grouped
Other                              14
La Tourette Parks & Golf Course     5
Van Cortlandt Park                  5
Prospect Park                       4
Forest Park                         3
High Rock Park                      2
Name: count, dtype: int64


---

## Summary

In this chapter, you learned:

1. **Nominal vs Ordinal** - Know your data types before encoding
2. **Binary/Label Encoding** - For binary or ordinal features
3. **One-Hot Encoding** - For nominal features with few categories
4. **Ordinal Encoding** - For features with meaningful order
5. **Frequency Encoding** - For high cardinality features
6. **Target Encoding** - Powerful but risky (data leakage)
7. **High Cardinality** - Group rare categories or use specialized encodings

### What's Next?

- **Chapter 5**: Feature Selection - Choose the most relevant features
- **Chapter 6**: Preprocessing Pipelines - Put it all together

---

## Exercises

### Exercise 1: Encode the Volunteers Dataset

Load `volunteers.csv` and:
1. Identify all categorical columns
2. Apply One-Hot Encoding to `category_desc`
3. Apply Ordinal Encoding to any ordinal features you find

In [18]:
# Your code here


### Exercise 2: Compare Encoding Impact on Model

Using the wine dataset:
1. Create a categorical version of `Alcohol` (Low, Medium, High based on quantiles)
2. Encode it using both One-Hot and Label Encoding
3. Train a LogisticRegression model with each encoding
4. Compare the accuracy scores

In [19]:
# Your code here


### Exercise 3: Handle High Cardinality

If `Park_Name` had 100 unique values:
1. What problems would One-Hot Encoding cause?
2. Implement Frequency Encoding as an alternative
3. Group categories with fewer than 3 samples into 'Rare'

In [20]:
# Your code here


---

## How this is used in real work

`OneHotEncoder(handle_unknown='ignore')` inside a `ColumnTransformer` is the default you will write hundreds of times. Target encoding is standard on high-cardinality columns in tabular competitions and in production, and it is the single most common source of quiet leakage in both.

## Where this goes next

**C5** cuts the columns you have just multiplied. **C6** makes the whole sequence safe.